In [ ]:
# 🎧 Detección de la Mejor Toma Musical con YAMNet

# 📁 Montar Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

# 📦 Instalar librerías necesarias
# !pip install -q tensorflow tensorflow_hub pydub librosa
# !apt install -y ffmpeg

# 🧠 Cargar modelo YAMNet
import tensorflow_hub as hub
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# 🔊 Funciones para procesar audio
import librosa
import numpy as np
import os
from pydub import AudioSegment
import pandas as pd
import tensorflow as tf

def load_mp3(file_path, sr=16000):
    audio = AudioSegment.from_mp3(file_path)
    audio = audio.set_channels(1).set_frame_rate(sr)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32) / 32768.0
    return samples

def score_take(waveform, sr=16000):
    try:
        # Beat tracking
        tempo, beats = librosa.beat.beat_track(y=waveform, sr=sr)
        beat_var = np.std(np.diff(beats)) if len(beats) > 2 else 100

        # RMS
        rms = librosa.feature.rms(y=waveform)[0]
        avg_rms = np.mean(rms)

        # Spectral centroid
        centroid = librosa.feature.spectral_centroid(y=waveform, sr=sr)[0]
        avg_centroid = np.mean(centroid)

        # Silence ratio
        intervals = librosa.effects.split(waveform, top_db=30)
        silence_ratio = 1 - sum(i[1]-i[0] for i in intervals) / len(waveform)

        # YAMNet embeddings and labels
        waveform_tf = tf.convert_to_tensor(waveform, dtype=tf.float32)
        scores, embeddings, spectrogram = yamnet_model(waveform_tf)
        mean_scores = tf.reduce_mean(scores, axis=0).numpy()

        # Penalizar presencia de 'noise', 'hum', 'buzz'
        penalty_labels = ["Hum", "Buzz", "Static", "Noise"]
        class_map_path = tf.keras.utils.get_file(
            'yamnet_class_map.csv',
            'https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv')
        df_labels = pd.read_csv(class_map_path)
        penalty_score = sum([mean_scores[i] for i, label in enumerate(df_labels.display_name) if label in penalty_labels])

        # Score final (ajustable)
        score = (avg_rms * 0.4) - (beat_var * 0.2) + (avg_centroid * 0.2) - (silence_ratio * 0.2) - (penalty_score * 5)
        return score
    except Exception as e:
        print(f"Error procesando toma: {e}")
        return -999

# 🗂️ Procesar carpetas
from pathlib import Path

#ensayo_root = "/content/drive/MyDrive/Ensayos"
ensayo_root = "/Users/javiercordero/workspace/jac/zapa-ia"
resultados = []

for carpeta in Path(ensayo_root).iterdir():
    if carpeta.is_dir():
        tomas = []
        for mp3_file in carpeta.glob("*.mp3"):
            print(f"Procesando: {mp3_file.name}")
            try:
                waveform = load_mp3(str(mp3_file))
                score = score_take(waveform)
                tomas.append({
                    "Ensayo": carpeta.name,
                    "Toma": mp3_file.name,
                    "Score": score
                })
                print(f"score: {score}")
            except Exception as e:
                print(f"❌ Error procesando {mp3_file.name}: {e}")
        
        # Ordenar tomas por score descendente
        top5 = sorted(tomas, key=lambda x: x["Score"], reverse=True)[:5]
        resultados.extend(top5)
        
print(resultados)
df = pd.DataFrame(resultados)
df = df.sort_values(by=["Ensayo", "Score"], ascending=[True, False])
df.reset_index(drop=True, inplace=True)
df.to_csv("ranking_top5_tomas_por_carpeta.csv", index=False)
print("✅ Archivo exportado como 'ranking_top5_tomas_por_carpeta.csv'")


Procesando: monkey doc  nebu temaso bass.mp3
Procesando: monkey doc  una lastima no me importa 2.mp3
Procesando: monkey doc  una lastima no me importa.mp3
Procesando: estado de nebulosa 2025 - nebulosa.mp3
Procesando: monkey doc intro 2025 - nebulosa.mp3
Procesando: amor - nebu 25.mp3
Procesando: e pasada 2 del abasto - nebu 25.mp3
Procesando: en la sala nebu 25.mp3
Procesando: monkey doc pobre 2025 - nebulosa.mp3
Procesando: quepollo y medio 2025 - nebulosa.mp3
Procesando: monkey doc guitar bass.mp3
[{'Ensayo': 'zapadas', 'Toma': 'quepollo y medio 2025 - nebulosa.mp3', 'Score': 267.51601676036563}, {'Ensayo': 'zapadas', 'Toma': 'en la sala nebu 25.mp3', 'Score': 257.3661326042635}, {'Ensayo': 'zapadas', 'Toma': 'e pasada 2 del abasto - nebu 25.mp3', 'Score': 241.71541536974513}, {'Ensayo': 'zapadas', 'Toma': 'monkey doc  una lastima no me importa.mp3', 'Score': 192.49536423398206}, {'Ensayo': 'zapadas', 'Toma': 'monkey doc intro 2025 - nebulosa.mp3', 'Score': 192.14316126435455}]
✅ Ar